In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import sys
import numpy as np
import librosa
import joblib
from tensorflow.keras.models import load_model



In [22]:
# ======================
# Feature Extraction
# (MUST match training)
# ======================
def extract_features(data, sr):
    result = np.array([])

    zcr = np.mean(librosa.feature.zero_crossing_rate(y=data).T, axis=0)
    result = np.hstack((result, zcr))

    stft = np.abs(librosa.stft(data))
    chroma = np.mean(librosa.feature.chroma_stft(S=stft, sr=sr).T, axis=0)
    result = np.hstack((result, chroma))

    mfcc = np.mean(librosa.feature.mfcc(y=data, sr=sr, n_mfcc=20).T, axis=0)
    result = np.hstack((result, mfcc))

    rms = np.mean(librosa.feature.rms(y=data).T, axis=0)
    result = np.hstack((result, rms))

    mel = np.mean(librosa.feature.melspectrogram(y=data, sr=sr).T, axis=0)
    result = np.hstack((result, mel))

    return result



In [23]:
def preprocess_audio(path):
    data, sr = librosa.load(path, duration=2.5, offset=0.6)
    features = extract_features(data, sr)
    return features.reshape(1, -1)


In [24]:
# Empathy Score Calculation
# ======================
# Example mapping of emotions to empathy scores (0-1 scale)
emotion_empathy_map = {
    "neutral": 0.2,
    "happy": 0.3,
    "sad": 0.9,
    "angry": 0.8,
    "fear": 0.7,
    "surprise": 0.4,
    "disgust": 0.5
}

def calculate_empathy(pred_probs, emotions):
    """Calculate empathy score for each emotion and overall empathy."""
    empathy_scores = {}
    total_score = 0
    for emotion, prob in zip(emotions, pred_probs):
        score = prob * emotion_empathy_map.get(emotion, 0.5)  # default 0.5 if not mapped
        empathy_scores[emotion] = score
        total_score += score
    overall_empathy = total_score
    return empathy_scores, overall_empathy


In [25]:
from google.colab import files # Import files for upload functionality

# MAIN
# ======================
print("Loading model and preprocessors...")

model = load_model("/content/drive/MyDrive/Colab Notebooks/FYP/audio_emotion_cnn.h5")
scaler = joblib.load("/content/drive/MyDrive/Colab Notebooks/FYP/scaler.pkl")
encoder = joblib.load("/content/drive/MyDrive/Colab Notebooks/FYP/encoder.pkl")

# ------------------------------
# Upload Audio File(s)
# ------------------------------
print("Upload your audio file(s) (mp3 or wav):")
uploaded = files.upload()  # Opens a file picker

# Assuming only one file is uploaded for processing in this context
import sys
if uploaded:
    audio_file_name = list(uploaded.keys())[0]
    audio_path = audio_file_name # Assign the uploaded file name as audio_path
    print("\n===================================")
    print(f"Processing file: {audio_file_name}")
else:
    print("No file uploaded. Exiting.")
    sys.exit(1) # Exit if no file is uploaded


Loading model and preprocessors...


Upload your audio file(s) (mp3 or wav):


Saving test1.aac to test1.aac

Processing file: test1.aac


In [26]:
# Preprocess audio
print("Processing audio file...")
features = preprocess_audio(audio_path)



Processing audio file...


/tmp/ipython-input-3464127453.py:2: UserWarning: PySoundFile failed. Trying audioread instead.
  data, sr = librosa.load(path, duration=2.5, offset=0.6)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [27]:
# Scale and reshape
features = scaler.transform(features)
features = np.expand_dims(features, axis=2)



In [28]:
# Predict
pred = model.predict(features)
predicted_label = encoder.inverse_transform(pred)[0][0]
confidence = np.max(pred) * 100



1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step


In [29]:
# Calculate empathy scores
emotions = encoder.categories_[0]
empathy_scores, overall_empathy = calculate_empathy(pred[0], emotions)



In [30]:
# OUTPUT
# ======================
print("\n🎤 Emotion Prediction Result")
print("----------------------------")
print(f"Dominant Emotion : {predicted_label}")
print(f"Confidence        : {confidence:.2f}%\n")

print("Emotion Probabilities:")
for emotion, prob in zip(emotions, pred[0]):
    print(f"{emotion:10s}: {prob*100:.2f}%")

print("\nEmpathy Scores per Emotion:")
for emotion, score in empathy_scores.items():
    print(f"{emotion:10s}: {score*100:.2f}%")

print(f"\n✅ Overall Empathy Score: {overall_empathy*100:.2f}%\n")


🎤 Emotion Prediction Result
----------------------------
Dominant Emotion : fear
Confidence        : 98.29%

Emotion Probabilities:
angry     : 0.02%
calm      : 0.00%
disgust   : 0.02%
fear      : 98.29%
happy     : 1.27%
neutral   : 0.00%
sad       : 0.39%
surprise  : 0.00%

Empathy Scores per Emotion:
angry     : 0.02%
calm      : 0.00%
disgust   : 0.01%
fear      : 68.80%
happy     : 0.38%
neutral   : 0.00%
sad       : 0.35%
surprise  : 0.00%

✅ Overall Empathy Score: 69.57%

